In [1]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


gender_mapping = {
    "Male": 0,
    "Female": 1,
    "Other": 2
}

stress_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

yesno_mapping = {
    "No": 0,
    "Yes": 1
}

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e8/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e8/test.csv")

for df in [train, test]:
    df["gender"] = df["gender"].map(gender_mapping)
    df["stress_level"] = df["stress_level"].map(stress_mapping)
    df["academic_work_impact"] = df["academic_work_impact"].map(yesno_mapping)
    
# for field in train.drop(columns=["id", "addicted_label"]):
#         train[field] = train[field].fillna(train[field].mean())

# for field in test.drop(columns=["id"]):
#         test[field] = test[field].fillna(test[field].mean())

for df in [train, test]:
    df["gender"] = df["gender"].map(gender_mapping)
    df["stress_level"] = df["stress_level"].map(stress_mapping)
    df["academic_work_impact"] = df["academic_work_impact"].map(yesno_mapping)

    # 1. proportion of day spent gaming + social media
    df["gaming_social_ratio_day"] = (
        df["gaming_hours"] + df["social_media_hours"]
    ) / 24.0

    # 2. screen time relative to sleep
    df["screen_sleep_ratio"] = (
        df["daily_screen_time_hours"] /
        (df["sleep_hours"] + 1e-6)
    )

    # # 3. gaming + social media relative to work
    # df["gaming_social_work_ratio"] = (
    #     df["gaming_hours"] + df["social_media_hours"]
    # ) / (df["work_hours"] + 1e-6)

    # 4. non-productive screen share
    df["gaming_social_screen_ratio"] = (
        df["gaming_hours"] + df["social_media_hours"]
    ) / (df["daily_screen_time_hours"] + 1e-6)

    # # 5. screen time pressure relative to sleep + work
    # df["screen_life_ratio"] = (
    #     df["daily_screen_time_hours"]
    #     / (df["sleep_hours"] + df["work_hours"] + 1e-6)
    # )
    
submission = pd.read_csv("/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv")

X = train.drop(columns=["id", "addicted_label"])
y = train["addicted_label"]

X_test = test.drop(columns=["id"])

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=67,
    stratify=y
)
model = XGBClassifier(

    n_estimators=5500, # max decision tree

    learning_rate=0.02, # lr

    max_depth=6, # how deep a decision tree goes

    min_child_weight=8, # if a decision tree's hessian value is under 8 recompute

    gamma=0.05, # check if total gain is greater than gamma

    subsample=0.85, # how many samples

    colsample_bytree=0.90, # how many features

    reg_alpha=0.02, # l1 reg

    reg_lambda=2.0, # l2 reg

    objective="binary:logistic", # bce

    eval_metric="auc",

    tree_method="hist", # efficiency by grouping results in histograms and choosing best

    # device="cpu", # run on gpu

    random_state=67, # reproduce rand state

    n_jobs=-1, # how many threads

    early_stopping_rounds=200
)

model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_train, y_train),
        (X_val, y_val)
    ],
    verbose=250
)

pred_val = model.predict_proba(X_val)[:, 1]

auc = roc_auc_score(y_val,pred_val)

print("\nValidation ROC AUC:", auc)

print("Best iteration:", model.best_iteration)

pred_test = model.predict_proba(X_test)[:, 1]

submission["addicted_label"] = pred_test

submission.to_csv("submission_xgb.csv", index=False)

print(submission.head())

[0]	validation_0-auc:0.91464	validation_1-auc:0.91372
[250]	validation_0-auc:0.94146	validation_1-auc:0.94049
[500]	validation_0-auc:0.95096	validation_1-auc:0.94933
[750]	validation_0-auc:0.95687	validation_1-auc:0.95459
[1000]	validation_0-auc:0.96058	validation_1-auc:0.95767
[1250]	validation_0-auc:0.96311	validation_1-auc:0.95957
[1500]	validation_0-auc:0.96492	validation_1-auc:0.96083
[1750]	validation_0-auc:0.96632	validation_1-auc:0.96166
[2000]	validation_0-auc:0.96747	validation_1-auc:0.96225
[2250]	validation_0-auc:0.96844	validation_1-auc:0.96268
[2500]	validation_0-auc:0.96937	validation_1-auc:0.96306
[2750]	validation_0-auc:0.97017	validation_1-auc:0.96332
[3000]	validation_0-auc:0.97088	validation_1-auc:0.96353
[3250]	validation_0-auc:0.97157	validation_1-auc:0.96373
[3500]	validation_0-auc:0.97223	validation_1-auc:0.96388
[3750]	validation_0-auc:0.97284	validation_1-auc:0.96400
[4000]	validation_0-auc:0.97342	validation_1-auc:0.96410
[4250]	validation_0-auc:0.97398	valid